# MATCHA-Bench: reproduce the baseline experiments

This notebook reruns the baseline pipeline, which reproduces the Chinese MentalBERT recipe on MATCHA:
1. **Pre-training (DAPT)** of `hfl/chinese-bert-wwm-ext` on MATCHA with three masking strategies:
   random (the control), the depression lexicon (the Chinese MentalBERT recipe), and a 520-word
   multi-disorder lexicon.
2. **Fine-tuning**, 3 seeds each, on the 7 MATCHA tasks (the baselines) and on the 5 public
   Chinese MentalBERT tasks (a sanity check against the published scores).
3. **TF-IDF baseline, results tables and significance tests.**

Code: https://github.com/shehryars715/matcha-bench (cloned automatically below).

### Setup (once)
1. **Settings → Accelerator: GPU T4 ×2**, and **Internet: On** (needs a phone-verified Kaggle account).
2. **Add Input → Datasets → `matcha-bench-data`** (private; access must be shared by the owner). The
   data is derived from MATCHA and covered by its Data Use Agreement.
3. Set `MODE` below: `"check"` runs a 5-minute sanity test; `"full"` runs the real pipeline.

### Time
The full pipeline takes about 80 GPU-hours on T4 ×2 (pre-training ≈ 11 h, fine-tuning ≈ 70 h). That is
far longer than one Kaggle session (12 h max) and more than Kaggle's weekly GPU quota (30 h). The notebook is
**resumable**: it stops cleanly before `BUDGET_HOURS` and saves its progress to the notebook output.
To continue, open the notebook's last version, **Add Input → Notebook output** (this notebook's
previous version), then *Save Version → Save & Run All* again. Finished work is skipped, and
pre-training resumes from its last checkpoint.

In [ ]:
MODE = "check"          # "check" (5-minute sanity test) or "full" (the real pipeline)
BUDGET_HOURS = 9.0      # stop cleanly before this many hours (Kaggle's limit is 12 h)
GIT_URL = "https://github.com/shehryars715/matcha-bench.git"
GIT_REF = "main"        # a commit hash or tag pins the exact code version

In [ ]:
import glob, os, shutil, subprocess, time
os.environ["JOB_START"] = str(time.time())   # the time budget counts from here
REPO = "/kaggle/working/matcha-bench"

def sh(cmd, cwd=REPO):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

!nvidia-smi --query-gpu=name,memory.total --format=csv

if not os.path.exists(REPO):
    sh(f"git clone --quiet {GIT_URL} {REPO}", cwd="/kaggle/working")
sh(f"git checkout --quiet {GIT_REF} && git log -1 --format='code version: %h %s'")
sh("pip install -q jieba")

In [ ]:
# Link the data (data/ and lexicons/) from the attached dataset.
hits = glob.glob("/kaggle/input/**/data/dapt", recursive=True)
assert hits, "Attach the matcha-bench-data dataset (Add Input → Datasets)"
data_root = os.path.dirname(os.path.dirname(hits[0]))
for name in ("data", "lexicons"):
    target = os.path.join(REPO, name)
    if os.path.islink(target):
        os.unlink(target)
    elif os.path.isdir(target):
        shutil.rmtree(target)
    os.symlink(os.path.join(data_root, name), target)
print("data:", data_root)

# Resume: restore outputs/ from a previous version of this notebook if it is attached.
for prev in glob.glob("/kaggle/input/**/matcha-bench/outputs", recursive=True):
    print("restoring progress from", prev)
    shutil.copytree(prev, os.path.join(REPO, "outputs"), dirs_exist_ok=True)

## Run

In [ ]:
if MODE == "check":
    # Tiny randomly initialised BERT: checks data, masking, training and fine-tuning end to end.
    sh("python scripts/run_dapt.py --config configs/dapt/smoke.yaml")
    sh("python scripts/run_finetune.py --tasks suicide --models outputs/dapt/smoke/final --seeds 42 "
       "--set defaults.epochs=1 output_root=outputs/check_finetune")
    print("\nCheck passed. Set MODE = 'full' to run the real pipeline.")
else:
    sh(f"python scripts/run_pipeline.py --budget-hours {BUDGET_HOURS}")

## Results
Tables appear once fine-tuning has produced results: mean ± std over 3 seeds, in percent.
`pipeline_log.jsonl` shows what each session did.

In [ ]:
import pandas as pd
log = os.path.join(REPO, "outputs/pipeline_log.jsonl")
if os.path.exists(log):
    print(open(log, encoding="utf-8").read()[-3000:])
for name in ("summary_f1.csv", "summary_macro_f1.csv"):
    path = os.path.join(REPO, "outputs/results", name)
    if os.path.exists(path):
        print(name); display(pd.read_csv(path, index_col=0))
for path in sorted(glob.glob(os.path.join(REPO, "outputs/results/significance_*.csv"))):
    print(os.path.basename(path)); display(pd.read_csv(path).groupby("task")[["delta", "p_value"]].mean())

In [ ]:
# Keep the saved output small: drop intermediate DAPT checkpoints once a final model exists.
for ckpt in glob.glob(os.path.join(REPO, "outputs/dapt/*/checkpoint-*")):
    if os.path.exists(os.path.join(os.path.dirname(ckpt), "final")):
        shutil.rmtree(ckpt)
!du -sh /kaggle/working/matcha-bench/outputs/* 2>/dev/null